# Exercise 7: From the log to the lake

Every run of your job does three things: **read** the new records of the topic `logs`,
**write** them into the lake (Parquet files in `data/lake/`), **remember** where it stopped.

This notebook gives you the building blocks (`tools/lakelib.py`). You write one function,
`run_once()`. Save your final version of this notebook into your `solution/` folder.

In [1]:
import sys; sys.path.append("/home/jovyan/tools")
from lakelib import *

(installing confluent-kafka==2.11.0 ...)
(installing duckdb==1.5.6 ...)


## 1. The example: a job that copies everything

It reads the topic from the beginning, every time. Run the next two cells and look at the
checker's last line.

In [2]:
def run_once():
    new = read_new(None)          # None = from the beginning of the topic
    write_files(new, LAKE)        # straight into the lake

reset()                           # empty lake
run_once()
check()

reset: lake, staging and state deleted
read_new: 2,000,000 new records
write_files: 2,000,000 records -> /home/jovyan/data/lake/logs/run-20261124-222418-fdeda0

Lake check — topic 'logs' (4 partitions)   2026-11-24 22:31:43
lake: read_parquet('/home/jovyan/data/lake/logs/**/*.parquet')

partition          topic offsets     in lake  duplicates    missing  not loaded
-------------------------------------------------------------------------------
        0           [0, 464,145)     464,145           0          0           0
        1           [0, 493,340)     493,340           0          0           0
        2           [0, 516,274)     516,274           0          0           0
        3           [0, 526,241)     526,241           0          0           0
-------------------------------------------------------------------------------
    total      2,000,000 records   2,000,000           0          0           0

files under /home/jovyan/data/lake: 4 data files, 111.7 MB, 27.93 MB pe

{'topic': 2000000,
 'lake': 2000000,
 'dup': 0,
 'miss': 0,
 'lag': 0,
 'files': 4,
 'mb': 111.707857,
 'ok': True,
 'problems': []}

In [3]:
run_once()                        # the next run
check()

read_new: 2,000,000 new records
write_files: 2,000,000 records -> /home/jovyan/data/lake/logs/run-20261124-223144-9d63dc


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))


Lake check — topic 'logs' (4 partitions)   2026-11-24 22:33:59
lake: read_parquet('/home/jovyan/data/lake/logs/**/*.parquet')

partition          topic offsets     in lake  duplicates    missing  not loaded
-------------------------------------------------------------------------------
        0           [0, 464,145)     928,290     464,145          0           0
        1           [0, 493,340)     986,680     493,340          0           0
        2           [0, 516,274)   1,032,548     516,274          0           0
        3           [0, 526,241)   1,052,482     526,241          0           0
-------------------------------------------------------------------------------
    total      2,000,000 records   4,000,000   2,000,000          0           0

files under /home/jovyan/data/lake: 8 data files, 223.4 MB, 27.93 MB per file on average
check took 5.7 s

Exactly once: FAIL — 2,000,000 duplicates


{'topic': 2000000,
 'lake': 4000000,
 'dup': 2000000,
 'miss': 0,
 'lag': 0,
 'files': 8,
 'mb': 223.415714,
 'ok': False,
 'problems': ['2,000,000 duplicates']}

## 2. The building blocks

| Block | What it does |
|---|---|
| `load_state()` | where the last run stopped, from a small state file (`None` if there is none) |
| `save_state(new.end)` | stores where this run stopped in the state file |
| `lake_position()` | where the last run stopped, read from the lake itself (`None` if the lake is empty) |
| `read_new(start)` | the records after `start`, up to the end of the topic *now* (`new.end`) |
| `write_files(new, folder)` | writes them as Parquet into a new run folder inside `LAKE` or `STAGING` |
| `publish(run)` | moves a finished run folder from `STAGING` into the lake: one rename, a single step |

`LAKE` is what readers see; `STAGING` is a folder next to it that nobody reads.
`new.df.show()` shows the records of a run as a Spark DataFrame.

## 3. Your architecture

Write `run_once()` for the architecture you chose (A, B or C, see the README).
The order of the calls *is* your architecture.

In [4]:
def run_once():
    # 1. where did the last run stop?
    start = lake_position()
    # 2. read what is new
    new = read_new(start)
    # 3. write it
    run = write_files(new, STAGING)
    # 4. remember where you stopped (if your architecture needs it)
    publish(run)

## 4. Check it

`check_all` empties the lake, then runs your job in five situations: a first run, new records,
nothing new, a run that stops **while** writing, a run that stops **after** writing.
It takes 2-4 minutes.

Architecture C: readers use a view, so check through it: `check_all(run_once, lake_sql=DEDUP_SQL)`.

In [5]:
check_all(run_once)

reset: lake, staging and state deleted


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))


situation                                 in lake  duplicates  missing  not loaded    run  check
-----------------------------------------------------------------------------------------------
1 first run                             2,000,000           0        0           0 128.4s  PASS
2 new records arrive, run again         2,100,000           0        0           0  27.8s  PASS
3 nothing new, run again                2,100,000           0        0           0   1.5s  PASS
4 run stops WHILE writing: right after  2,100,000           0        0     100,000         PASS
  ... then the next run                 2,200,000           0        0           0  28.1s  PASS
5 run stops AFTER writing: right after  2,200,000           0        0     100,000         PASS
  ... then the next run                 2,300,000           0        0           0  29.7s  PASS
-----------------------------------------------------------------------------------------------
Exactly once after every run: YES


[('1 first run',
  {'topic': 2000000,
   'lake': 2000000,
   'dup': 0,
   'miss': 0,
   'lag': 0,
   'files': 4,
   'mb': 111.707857,
   'ok': True,
   'problems': []},
  128.36887121200562),
 ('2 new records arrive, run again',
  {'topic': 2100000,
   'lake': 2100000,
   'dup': 0,
   'miss': 0,
   'lag': 0,
   'files': 8,
   'mb': 112.864553,
   'ok': True,
   'problems': []},
  27.829946279525757),
 ('3 nothing new, run again',
  {'topic': 2100000,
   'lake': 2100000,
   'dup': 0,
   'miss': 0,
   'lag': 0,
   'files': 8,
   'mb': 112.864553,
   'ok': True,
   'problems': []},
  1.5162012577056885),
 ('4 run stops WHILE writing: right after',
  {'topic': 2200000,
   'lake': 2100000,
   'dup': 0,
   'miss': 0,
   'lag': 100000,
   'files': 8,
   'mb': 112.864553,
   'ok': True,
   'problems': []},
  None),
 ('  ... then the next run',
  {'topic': 2200000,
   'lake': 2200000,
   'dup': 0,
   'miss': 0,
   'lag': 0,
   'files': 12,
   'mb': 114.023277,
   'ok': True,
   'problems': []},